# Conversational RAG — Solution Notebook

**Module 05 · JHU Agentic AI Certificate Program**

---

## What Is Conversational RAG?

Standard RAG treats every user query independently — it retrieves documents and answers the question, but forgets everything once the response is returned.

**Conversational RAG** extends this by maintaining **chat history** across turns. This enables:
- Follow-up questions that reference earlier answers ("What are *its* advantages?")
- Contextual coreference resolution ("*it*", "*they*", "*that approach*")
- Multi-turn dialogue over a document corpus

The key architectural addition is a **history-aware retriever** that reformulates ambiguous follow-up questions into self-contained queries before hitting the vector store.

## Learning Objectives

By the end of this notebook you will be able to:
1. Build a **history-aware retriever** using `create_history_aware_retriever`
2. Compose a full **conversational retrieval chain** using `create_retrieval_chain`
3. Manage **per-session message history** with `RunnableWithMessageHistory`
4. Test multi-turn conversations where follow-up questions rely on prior context

## Module Versions
| Package | Version |
|---|---|
| `langchain` | 0.3.27 |
| `langchain-core` | 0.3.79 |
| `langchain-openai` | 0.3.11 |
| `langchain-community` | 0.3.31 |
| `chromadb` | 1.3.4 |
| `tiktoken` | 0.12.0 |

---
## Part 1 — Installation

In [ ]:
!pip install \
    langchain==0.3.27 \
    langchain-core==0.3.79 \
    langchain-openai==0.3.11 \
    langchain-community==0.3.31 \
    chromadb==1.3.4 \
    tiktoken==0.12.0 \
    --quiet

---
## Part 2 — Imports

In [ ]:
import json
import os
import warnings
warnings.filterwarnings('ignore')

# LangChain core
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# LangChain chains
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# LangChain OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Vector store
from langchain_community.vectorstores import Chroma

# Text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chat message history (in-memory store)
from langchain_community.chat_message_histories import ChatMessageHistory

print('All imports successful.')

---
## Part 3 — Configuration

We load API credentials from `config.json` (same pattern used throughout the course).

Expected `config.json` structure:
```json
{
  "API_KEY": "sk-...",
  "OPENAI_API_BASE": "https://api.openai.com/v1"
}
```

In [ ]:
file_name = 'config.json'
with open(file_name, 'r') as file:
    config = json.load(file)
    os.environ['OPENAI_API_KEY'] = config.get('API_KEY')
    os.environ['OPENAI_BASE_URL'] = config.get('OPENAI_API_BASE')

print('Configuration loaded.')

---
## Part 4 — Initialize LLM & Embeddings

In [ ]:
# Language model used for both question reformulation and answer generation
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,
    max_tokens=500
)

# Embeddings model used to encode documents and queries into vectors
embeddings = OpenAIEmbeddings(model='text-embedding-ada-002')

print(f'LLM   : {llm.model_name}')
print(f'Embed : text-embedding-ada-002')

---
## Part 5 — Sample Documents

We use in-memory `Document` objects about **renewable energy** (consistent with the Week-6 theme) so no external files are required.

In [ ]:
raw_documents = [
    Document(
        page_content=(
            "Solar energy harnesses sunlight using photovoltaic (PV) cells or concentrated solar power (CSP) systems. "
            "PV cells convert photons directly into electricity via the photovoltaic effect. "
            "Key advantages include zero operational emissions, low maintenance costs, and rapidly declining installation prices. "
            "Utility-scale solar farms can now generate electricity at costs competitive with fossil fuels in many regions. "
            "The main challenges are intermittency (no generation at night or in heavy cloud cover) and the land area required for large installations."
        ),
        metadata={'source': 'solar', 'year': 2024}
    ),
    Document(
        page_content=(
            "Wind energy converts the kinetic energy of moving air into electricity using wind turbines. "
            "Onshore wind is one of the cheapest sources of new electricity generation globally. "
            "Offshore wind turbines benefit from stronger, more consistent winds and have fewer land-use conflicts, "
            "but installation and maintenance costs are significantly higher. "
            "Wind power also faces intermittency challenges and can have local environmental impacts on birds and bats."
        ),
        metadata={'source': 'wind', 'year': 2024}
    ),
    Document(
        page_content=(
            "Hydroelectric power generates electricity by capturing the energy of flowing or falling water. "
            "It is the world's largest source of renewable electricity, accounting for roughly 16% of global power generation. "
            "Large dams provide reliable baseload power and grid-balancing services, but cause significant ecological disruption, "
            "including habitat loss and changes to river sediment flows. "
            "Run-of-river systems have a smaller environmental footprint but generate less power."
        ),
        metadata={'source': 'hydro', 'year': 2024}
    ),
    Document(
        page_content=(
            "Geothermal energy taps heat stored in the Earth's crust to generate electricity or provide direct heating. "
            "Unlike solar and wind, geothermal plants operate continuously (high capacity factor), making them well-suited for baseload generation. "
            "Viable sites are mostly located near tectonic plate boundaries, limiting widespread deployment. "
            "Enhanced geothermal systems (EGS) aim to unlock geothermal potential in areas without natural hydrothermal resources."
        ),
        metadata={'source': 'geothermal', 'year': 2024}
    ),
    Document(
        page_content=(
            "Tidal and wave energy systems capture kinetic and potential energy from ocean currents and waves. "
            "Tidal energy is highly predictable because it depends on gravitational forces from the Moon and Sun. "
            "Wave energy converters are still largely pre-commercial, with high capital costs and durability challenges in marine environments. "
            "The UK, South Korea, and Canada lead in tidal barrage and tidal stream development."
        ),
        metadata={'source': 'tidal', 'year': 2024}
    ),
    Document(
        page_content=(
            "Biomass energy uses organic material — wood, agricultural residues, and dedicated energy crops — as fuel. "
            "When managed sustainably, biomass can be carbon-neutral because the CO2 released during combustion is offset by CO2 absorbed during plant growth. "
            "Bioenergy with Carbon Capture and Storage (BECCS) is considered a key negative-emissions technology in climate models. "
            "However, large-scale biomass use raises concerns about land competition, biodiversity loss, and actual lifecycle emissions."
        ),
        metadata={'source': 'biomass', 'year': 2024}
    ),
    Document(
        page_content=(
            "Green hydrogen is produced by using renewable electricity to split water molecules via electrolysis. "
            "It serves as an energy carrier for sectors that are hard to electrify directly, such as steel manufacturing and long-haul shipping. "
            "Current limitations include the high cost of electrolysers and the significant energy losses in the production-storage-use chain. "
            "Falling electrolyser costs and growing renewable capacity are expected to make green hydrogen cost-competitive by 2030 in favorable locations."
        ),
        metadata={'source': 'hydrogen', 'year': 2024}
    ),
    Document(
        page_content=(
            "Grid-scale energy storage — primarily lithium-ion batteries — is critical for integrating variable renewable sources. "
            "Battery storage smooths supply-demand imbalances by storing excess solar or wind energy and discharging when generation is low. "
            "Battery costs have fallen over 97% since 1991 and continue to decline. "
            "Alternative long-duration storage technologies include pumped hydro, iron-air batteries, compressed air, and flow batteries."
        ),
        metadata={'source': 'storage', 'year': 2024}
    ),
    Document(
        page_content=(
            "Smart grids use digital communication technology to detect and react to local changes in electricity use. "
            "They enable bidirectional energy flows, real-time demand response, and better integration of distributed renewable sources. "
            "Key components include smart meters, sensors, automated switches, and AI-driven control systems. "
            "Smart grids improve efficiency, reduce outages, and support electric vehicle charging infrastructure."
        ),
        metadata={'source': 'smart_grid', 'year': 2024}
    ),
    Document(
        page_content=(
            "Carbon capture and storage (CCS) captures CO2 emissions at the source — such as a power plant or industrial facility — "
            "and stores them underground in geological formations. "
            "Direct air capture (DAC) goes further by removing CO2 directly from the atmosphere. "
            "Both technologies face challenges of high energy consumption and cost, but are considered essential for meeting 1.5°C climate targets "
            "in scenarios where deep decarbonization of certain sectors proves difficult."
        ),
        metadata={'source': 'ccs', 'year': 2024}
    ),
]

print(f'Loaded {len(raw_documents)} documents.')

---
## Part 6 — Text Splitting & Vector Store

In [ ]:
# Split documents into smaller chunks for more precise retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(raw_documents)
print(f'Split into {len(chunks)} chunks.')

In [ ]:
# Create a Chroma vector store from the document chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='renewable_energy'
)

# Create a base retriever — returns top-3 most relevant chunks per query
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

print('Vector store and retriever ready.')

---
## Part 7 — History-Aware Retriever

The **history-aware retriever** wraps the base retriever with an LLM step that reformulates ambiguous follow-up questions into self-contained queries.

**Example:**
- Chat history: *"Tell me about solar energy."*
- Follow-up: *"What are its main advantages?"*
- Reformulated: *"What are the main advantages of solar energy?"* ← this is what hits the vector store

```
User question + chat history
         │
         ▼
  ┌─────────────┐
  │  LLM        │  (reformulates question)
  └──────┬──────┘
         │ standalone question
         ▼
  ┌─────────────┐
  │  Retriever  │  (searches vector store)
  └──────┬──────┘
         │ relevant chunks
         ▼
    Answer chain
```

In [ ]:
# System prompt that instructs the LLM how to reformulate follow-up questions
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question which might reference "
    "context in the chat history, formulate a standalone question which can be "
    "understood without the chat history. Do NOT answer the question, just "
    "reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ('system', contextualize_q_system_prompt),
    MessagesPlaceholder('chat_history'),   # chat history injected here
    ('human', '{input}'),                  # current user question
])

# Wrap the base retriever with history-aware question reformulation
history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt
)

print('History-aware retriever created.')

---
## Part 8 — Question-Answer Chain

This chain takes the retrieved documents plus the current question (and history) and generates a final answer.

In [ ]:
qa_system_prompt = (
    "You are an assistant for question-answering tasks about renewable energy. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ('system', qa_system_prompt),
    MessagesPlaceholder('chat_history'),   # history also available for answering
    ('human', '{input}'),
])

# Stuffs all retrieved documents into the context window for the LLM
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

print('QA chain created.')

---
## Part 9 — Assemble the Full Conversational Chain

`create_retrieval_chain` combines the history-aware retriever with the QA chain.
`RunnableWithMessageHistory` wraps the chain to automatically load/save per-session message history.

In [ ]:
# Combine history-aware retriever + QA chain into a single retrieval chain
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

# In-memory session store: session_id -> ChatMessageHistory
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Return (creating if needed) the message history for a session."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Wrap the chain so it automatically manages chat history per session
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history',
    output_messages_key='answer',
)

print('Conversational RAG chain assembled.')

---
## Part 10 — Multi-Turn Conversation Test

We run three questions in the same session. Questions 2 and 3 are deliberately ambiguous — they rely on the conversation history to be answered correctly.

In [ ]:
def ask(question: str, session_id: str = 'session_1') -> str:
    """Send a question to the conversational RAG chain and return the answer."""
    result = conversational_rag_chain.invoke(
        {'input': question},
        config={'configurable': {'session_id': session_id}}
    )
    return result['answer']

In [ ]:
# Turn 1 — establish the topic
q1 = 'What is solar energy and how does it work?'
a1 = ask(q1)
print(f'Q1: {q1}')
print(f'A1: {a1}')
print()

In [ ]:
# Turn 2 — pronoun "its" refers to solar energy from the previous turn
q2 = 'What are its main advantages?'
a2 = ask(q2)
print(f'Q2: {q2}')
print(f'A2: {a2}')
print()

In [ ]:
# Turn 3 — follow-up comparison requiring context from turns 1 and 2
q3 = 'How does it compare to wind energy in terms of cost and intermittency?'
a3 = ask(q3)
print(f'Q3: {q3}')
print(f'A3: {a3}')
print()

---
## Part 11 — Inspect Session History

We can inspect what has been stored in memory for the session.

In [ ]:
session_history = store['session_1']

print(f'Total messages stored: {len(session_history.messages)}')
print()
for i, msg in enumerate(session_history.messages):
    role = 'Human' if isinstance(msg, HumanMessage) else 'AI'
    print(f'[{i+1}] {role}: {msg.content[:120]}...')

---
## Part 12 — Multiple Sessions (Bonus)

Each `session_id` is isolated — different users or conversations do not share history.

In [ ]:
# New independent session about a different topic
q_new = 'What is green hydrogen and why is it important?'
a_new = ask(q_new, session_id='session_2')
print(f'Session 2 — Q: {q_new}')
print(f'Session 2 — A: {a_new}')
print()

# Follow up in session_2 only
q_followup = 'What are the current barriers to scaling it up?'
a_followup = ask(q_followup, session_id='session_2')
print(f'Session 2 follow-up — Q: {q_followup}')
print(f'Session 2 follow-up — A: {a_followup}')
print()

# Confirm session_1 is unaffected
print(f'Session 1 message count: {len(store["session_1"].messages)}')
print(f'Session 2 message count: {len(store["session_2"].messages)}')

---
## Summary

| Component | Role |
|---|---|
| `RecursiveCharacterTextSplitter` | Splits documents into chunks for retrieval |
| `Chroma` vector store | Stores and searches chunk embeddings |
| `create_history_aware_retriever` | Reformulates follow-up questions using chat history |
| `create_stuff_documents_chain` | Generates answers from retrieved context |
| `create_retrieval_chain` | Wires retriever + QA chain together |
| `RunnableWithMessageHistory` | Manages per-session message history automatically |
| `ChatMessageHistory` / `store` | In-memory storage for conversation turns |

**Key insight:** The history-aware retriever is what makes follow-up questions work. Without it, a query like *"What are its advantages?"* would retrieve documents about completely unrelated topics.